# 10 — Final Evaluation: Held-Out Test Set

**Purpose.** Scores the models developed in notebooks 08 (logistic
regression) and 09 (XGBoost) against the held-out test window
(2022-01-01 through 2026-03-31), the first and only time this project
touches test-period data. Reports ROC-AUC (primary) and PR-AUC (secondary)
against the test window's own documented 8.15% positive rate (not the
full-history 5.59% rate), states the result against every Success Criteria
threshold from `docs/phase0_framing.md`, and runs the per-window
data-quality gate on the test window's own observed positive rate.

**Refuses to run without both prior notebooks' locked/persisted results.**
Per this project's baseline lock-in discipline, this notebook stops rather
than silently proceeding if `outputs/baseline_lock.json`,
`outputs/modeling_sequence_logreg.json`, or `outputs/modeling_sequence_xgboost.json`
is missing.

**"Touching the test set" means any scoring here, and nowhere else in this
project up to this point.** Every prior notebook (05 through 09) worked
only within the training period; this is deliberate, not incidental.

**Business question (from `docs/phase0_framing.md`).**

> Can we predict, one quarter ahead, which US counties will see a housing
> price decline greater than 2%, and how much predictive power does
> mortgage-rate movement (30-year fixed rate level and rate-of-change) add
> once a county's own price momentum is already known?

This notebook is where that question gets its final, held-out answer.


## 1. Setup

In [1]:
import json
import os

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import text

from housing_risk.config import get_engine, TRAIN_END, TEST_START, TEST_END
from housing_risk import labels, features

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Refuse to run without prior notebooks' results

Per this project's baseline lock-in discipline (mirrored from notebook 08's
own lock-once behavior), this notebook does not proceed if any of the three
required result files is missing.


In [2]:
BASELINE_LOCK_PATH = "../outputs/baseline_lock.json"
LOGREG_RESULTS_PATH = "../outputs/modeling_sequence_logreg.json"
XGBOOST_RESULTS_PATH = "../outputs/modeling_sequence_xgboost.json"
TRAIN_FEATURES_PATH = "../data/processed/train_features.csv"

for path, note in [
    (BASELINE_LOCK_PATH, "run notebooks/08_modeling_sequence.ipynb first (it locks the baseline)."),
    (LOGREG_RESULTS_PATH, "run notebooks/08_modeling_sequence.ipynb first (it persists this file)."),
    (XGBOOST_RESULTS_PATH, "run notebooks/09_robustness_second_algo.ipynb first (it persists this file)."),
    (TRAIN_FEATURES_PATH, "run notebooks/07_feature_engineering.ipynb first (it persists this file)."),
]:
    assert os.path.exists(path), f"{path} not found -- {note} This notebook refuses to run without it."

with open(BASELINE_LOCK_PATH) as fh:
    baseline_lock = json.load(fh)
with open(LOGREG_RESULTS_PATH) as fh:
    logreg_results = json.load(fh)
with open(XGBOOST_RESULTS_PATH) as fh:
    xgboost_results = json.load(fh)

train_matrix = pd.read_csv(TRAIN_FEATURES_PATH, parse_dates=["month"])
print("All required prior-notebook outputs found. Proceeding to test-set scoring.")
print(f"Locked baseline pooled ROC-AUC (logistic regression): {baseline_lock['pooled_roc_auc']:.4f}")


All required prior-notebook outputs found. Proceeding to test-set scoring.
Locked baseline pooled ROC-AUC (logistic regression): 0.8873


## 3. Build the test-window feature matrix

Loads county-month rows through `TEST_END` (train + embargo + test -- the
full history needed so lag features computed *within* the test window still
have their own prior history available, including from the embargoed
quarters), then builds features/labels the same way notebook 07 did for
training, and finally restricts to test-window rows only
(`TEST_START` to `TEST_END`). This is the first and only notebook in this
project to load embargo- or test-period rows.

**The same trailing-edge exclusion notebook 07 has applies here too, and it's
the reason the test-window numbers below differ slightly from
`docs/phase0_framing.md`'s originally documented figures.** Because features
are deliberately shifted to predict one quarter ahead, the single most
recent quarter-end in the data loaded here (through `TEST_END`, 2026-03-31)
has no *later* quarter's price data available to compute its label from --
that data would fall after `TEST_END`, outside this project's declared test
window, and is correctly never loaded. So that last quarter-end's rows drop
out of `test_matrix` entirely. `docs/phase0_framing.md`'s original 52,087-row,
8.15%-positive test-window figures were computed before the one-quarter-ahead
alignment fix (see `src/housing_risk/features.py`'s docstring and notebook
06's superseded Section 5 for that history), on a same-quarter join that
didn't have this exclusion -- so a small, expected gap between those original
figures and what Section 4 below observes is normal, not a data problem.


In [3]:
engine = get_engine()
with engine.connect() as conn:
    has_table = conn.execute(text("SELECT to_regclass('county_month_check') IS NOT NULL")).scalar()
    assert has_table, "county_month_check doesn't exist -- run 03_join_sql_load.ipynb first."
    query = (
        "SELECT county_fips, month, zhvi, mortgage_rate FROM county_month_check "
        "WHERE month::date <= %(test_end)s"  # month is TEXT; cast before comparing
    )
    full_county_month = pd.read_sql(query, conn, params={"test_end": TEST_END})
full_county_month["month"] = pd.to_datetime(full_county_month["month"])

assert full_county_month["month"].max() <= TEST_END, (
    f"filter failed: max month {full_county_month['month'].max()} > TEST_END {TEST_END}"
)
print(f"{len(full_county_month):,} county-month rows through TEST_END ({full_county_month['month'].max().date()})")

mortgage_monthly_full = (
    full_county_month.groupby("month")["mortgage_rate"]
    .median()
    .reset_index()
    .sort_values("month")
    .reset_index(drop=True)
)

full_result = features.build_feature_matrix(full_county_month, mortgage_monthly_full)
full_matrix = full_result["matrix"]
print(
    f"Rows dropped project-wide for no one-quarter-later label (trailing edge of "
    f"the TEST_END-bounded data loaded above): {full_result['n_dropped_no_future_label']:,} "
    "-- this is what shifts the test-window figures below slightly from "
    "docs/phase0_framing.md's originally documented ones (see the note above)."
)

test_matrix = full_matrix.loc[
    (full_matrix["month"] >= TEST_START) & (full_matrix["month"] <= TEST_END)
].copy()

assert test_matrix["month"].min() >= TEST_START and test_matrix["month"].max() <= TEST_END, (
    "STOP: test_matrix has rows outside the TEST_START/TEST_END window -- leakage check failed."
)
print(f"Test-window feature matrix: {len(test_matrix):,} rows, {test_matrix['county_fips'].nunique():,} counties")


694,641 county-month rows through TEST_END (2026-03-31)
Rows dropped project-wide for no one-quarter-later label (trailing edge of the TEST_END-bounded data loaded above): 3,712 -- this is what shifts the test-window figures below slightly from docs/phase0_framing.md's originally documented ones (see the note above).
Test-window feature matrix: 48,960 rows, 3,071 counties


## 4. Per-window data-quality gate on the test window

Per `docs/phase0_framing.md`'s Model Validation Requirements, this compares
the test window's own observed positive rate against its *own* documented
figure (8.15%), never a blanket tolerance shared with the training window's
different (4.96%) rate. A small gap here is expected, for the reason named in
Section 3 above (the trailing test quarter drops out under the one-quarter-
ahead label alignment) -- the tolerance below accounts for that, rather than
treating any nonzero gap as a failure.


In [4]:
observed_test_rate = 100 * test_matrix["decline_label"].mean()
observed_test_rows = len(test_matrix)
expected_test_rate = 8.15
expected_test_rows = 52_087

print(f"Test-window rows: {observed_test_rows:,} (docs/phase0_framing.md: {expected_test_rows:,})")
print(f"Test-window positive rate: {observed_test_rate:.2f}% (docs/phase0_framing.md: {expected_test_rate:.2f}%)")

rate_diff = abs(observed_test_rate - expected_test_rate)
if rate_diff < 1.5:
    print(f"Test-window positive-rate gate PASSED (within 1.5 points of the documented 8.15% test-window rate).")
else:
    print(
        f"NOTE: test-window positive rate differs from docs/phase0_framing.md's documented 8.15% "
        f"by {rate_diff:.2f} points -- this is checked against the TEST window's own documented rate "
        "specifically (not the training window's 4.96% or the full-history 5.59%), per the per-window "
        "gate requirement. Investigate before trusting the scores below if this gap is large."
    )


Test-window rows: 48,960 (docs/phase0_framing.md: 52,087)
Test-window positive rate: 8.53% (docs/phase0_framing.md: 8.15%)
Test-window positive-rate gate PASSED (within 1.5 points of the documented 8.15% test-window rate).


## 5. Refit both models on the full training period, score on test

Both the logistic-regression and XGBoost models are refit once on the
*entire* training-period feature matrix (all of `train_matrix`, not a CV
fold slice -- this is the final model each algorithm would actually be
deployed with) using the step-3 (full) feature set, then scored on the
untouched test matrix. This is the only time in this project a model is
trained on the complete training window and scored on test data.


In [5]:
FEATURE_COLUMNS = features.FEATURE_COLUMNS
TARGET = "decline_label"

logreg_final = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
])
logreg_final.fit(train_matrix[FEATURE_COLUMNS], train_matrix[TARGET])
logreg_test_pred = logreg_final.predict_proba(test_matrix[FEATURE_COLUMNS])[:, 1]

xgb_final = xgb.XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="auc",
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_final.fit(train_matrix[FEATURE_COLUMNS], train_matrix[TARGET])
xgb_test_pred = xgb_final.predict_proba(test_matrix[FEATURE_COLUMNS])[:, 1]

y_test = test_matrix[TARGET].to_numpy()

final_scores = {}
for name, pred in [("logistic_regression", logreg_test_pred), ("xgboost", xgb_test_pred)]:
    roc = roc_auc_score(y_test, pred)
    pr = average_precision_score(y_test, pred)
    final_scores[name] = {"roc_auc": float(roc), "pr_auc": float(pr)}
    print(f"{name}: test ROC-AUC = {roc:.4f}, test PR-AUC = {pr:.4f} "
          f"(test-window baseline prevalence: {y_test.mean():.4f})")


logistic_regression: test ROC-AUC = 0.8614, test PR-AUC = 0.4797 (test-window baseline prevalence: 0.0853)
xgboost: test ROC-AUC = 0.8747, test PR-AUC = 0.4466 (test-window baseline prevalence: 0.0853)


## 6. Test-set marginal deltas, feature step by feature step

Refits each feature step (baseline / step 2 / step 3) on the full training
period and scores on the untouched test set, for both algorithms, so the
same three-step comparison used throughout this project gets its final,
held-out read -- not just the full-feature-set numbers from Section 5.


In [6]:
MOMENTUM_FEATURES = ["momentum_1m", "momentum_3m", "momentum_6m"]
RATE_LEVEL_FEATURES = ["mortgage_rate_level"]
RATE_CHANGE_FEATURES = ["mortgage_rate_roc", "rate_roc_x_momentum"]
FEATURE_STEPS = {
    "baseline": MOMENTUM_FEATURES,
    "step2_rate_level": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES,
    "step3_rate_change": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES + RATE_CHANGE_FEATURES,
}

def fit_and_score(make_model_fn, feature_cols):
    model = make_model_fn()
    model.fit(train_matrix[feature_cols], train_matrix[TARGET])
    pred = model.predict_proba(test_matrix[feature_cols])[:, 1]
    return {
        "roc_auc": float(roc_auc_score(y_test, pred)),
        "pr_auc": float(average_precision_score(y_test, pred)),
    }, pred

def make_logreg():
    return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))])

def make_xgb():
    return xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="auc",
        random_state=RANDOM_STATE, n_jobs=-1,
    )

test_step_scores = {"logistic_regression": {}, "xgboost": {}}
test_step_preds = {"logistic_regression": {}, "xgboost": {}}
for step_name, cols in FEATURE_STEPS.items():
    lr_score, lr_pred = fit_and_score(make_logreg, cols)
    xg_score, xg_pred = fit_and_score(make_xgb, cols)
    test_step_scores["logistic_regression"][step_name] = lr_score
    test_step_scores["xgboost"][step_name] = xg_score
    test_step_preds["logistic_regression"][step_name] = lr_pred
    test_step_preds["xgboost"][step_name] = xg_pred
    print(f"{step_name}: logreg test ROC-AUC={lr_score['roc_auc']:.4f}, "
          f"xgboost test ROC-AUC={xg_score['roc_auc']:.4f}")


baseline: logreg test ROC-AUC=0.8640, xgboost test ROC-AUC=0.8822
step2_rate_level: logreg test ROC-AUC=0.8648, xgboost test ROC-AUC=0.8731
step3_rate_change: logreg test ROC-AUC=0.8614, xgboost test ROC-AUC=0.8747


## 7. Quarter-clustered bootstrap CIs on test-set marginal deltas

Same method as notebooks 08/09, applied here to the test-set predictions
instead of CV out-of-fold predictions: resample whole test-window quarters
with replacement, keep every row for a chosen quarter together, reuse the
same resampled quarters for both models in a paired comparison.


In [7]:
N_BOOTSTRAP = 2000
rng = np.random.default_rng(RANDOM_STATE)

def quarter_clustered_bootstrap_delta(months, y_true, pred_a, pred_b, n_bootstrap=N_BOOTSTRAP, rng=rng):
    df = pd.DataFrame({"month": months, "y_true": y_true, "pred_a": pred_a, "pred_b": pred_b})
    quarters = df["month"].unique()
    by_quarter = {q: df.loc[df["month"] == q] for q in quarters}

    deltas = []
    for _ in range(n_bootstrap):
        draw = rng.choice(quarters, size=len(quarters), replace=True)
        resampled = pd.concat([by_quarter[q] for q in draw], ignore_index=True)
        if resampled["y_true"].nunique() < 2:
            continue
        auc_a = roc_auc_score(resampled["y_true"], resampled["pred_a"])
        auc_b = roc_auc_score(resampled["y_true"], resampled["pred_b"])
        deltas.append(auc_b - auc_a)

    deltas = np.array(deltas)
    point_delta = roc_auc_score(y_true, pred_b) - roc_auc_score(y_true, pred_a)
    ci_low, ci_high = float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))
    significant = not (ci_low <= 0.0 <= ci_high)
    return {
        "point_delta": float(point_delta), "ci_95_low": ci_low, "ci_95_high": ci_high,
        "n_bootstrap_used": int(len(deltas)), "significant": bool(significant),
    }

test_months = test_matrix["month"].to_numpy()
final_deltas = {}
for algo in ("logistic_regression", "xgboost"):
    d2 = quarter_clustered_bootstrap_delta(
        test_months, y_test,
        test_step_preds[algo]["baseline"], test_step_preds[algo]["step2_rate_level"],
    )
    d3 = quarter_clustered_bootstrap_delta(
        test_months, y_test,
        test_step_preds[algo]["step2_rate_level"], test_step_preds[algo]["step3_rate_change"],
    )
    final_deltas[algo] = {"step2_vs_baseline": d2, "step3_vs_step2": d3}
    print(f"{algo}:")
    print(f"  step 2 vs baseline: {d2['point_delta']:+.4f}  95% CI [{d2['ci_95_low']:+.4f}, {d2['ci_95_high']:+.4f}]")
    print(f"  step 3 vs step 2:   {d3['point_delta']:+.4f}  95% CI [{d3['ci_95_low']:+.4f}, {d3['ci_95_high']:+.4f}]")


logistic_regression:
  step 2 vs baseline: +0.0008  95% CI [-0.0027, +0.0039]
  step 3 vs step 2:   -0.0034  95% CI [-0.0222, +0.0186]
xgboost:
  step 2 vs baseline: -0.0092  95% CI [-0.0191, -0.0003]
  step 3 vs step 2:   +0.0016  95% CI [-0.0077, +0.0102]


## 8. Final result against every Success Criteria threshold

Per `docs/phase0_framing.md`'s Success Criteria table:

- Step 2 (+ rate level): ROC-AUC >= baseline + 0.01.
- Step 3 (+ rate-of-change/interaction): ROC-AUC >= step 2 + 0.01, or a
  stated null result.
- Robustness: same ranking of contributions across >= 2 algorithm families.

A delta whose 95% CI includes zero is stated as no significant marginal
contribution, not rounded to a small positive result.


In [8]:
def describe_delta(label, delta_result, threshold=0.01):
    meets_point = delta_result["point_delta"] >= threshold
    if not delta_result["significant"]:
        verdict = "NO SIGNIFICANT MARGINAL CONTRIBUTION (95% CI includes zero)"
    elif meets_point:
        verdict = f"significant, clears the +{threshold:.2f} threshold"
    else:
        verdict = f"significant, but below the +{threshold:.2f} threshold"
    print(f"{label}: delta {delta_result['point_delta']:+.4f}, "
          f"95% CI [{delta_result['ci_95_low']:+.4f}, {delta_result['ci_95_high']:+.4f}] -> {verdict}")

for algo in ("logistic_regression", "xgboost"):
    print(f"\n=== {algo} ===")
    print(f"Test ROC-AUC by step: " + ", ".join(
        f"{s}={test_step_scores[algo][s]['roc_auc']:.4f}" for s in FEATURE_STEPS
    ))
    print(f"Test PR-AUC by step:  " + ", ".join(
        f"{s}={test_step_scores[algo][s]['pr_auc']:.4f}" for s in FEATURE_STEPS
    ) + f"  (test-window baseline prevalence: {y_test.mean():.4f}, i.e. the PR-AUC floor for this window)")
    describe_delta("  Step 2 vs baseline", final_deltas[algo]["step2_vs_baseline"])
    describe_delta("  Step 3 vs step 2  ", final_deltas[algo]["step3_vs_step2"])

logreg_ranking = (
    final_deltas["logistic_regression"]["step2_vs_baseline"]["significant"]
    and final_deltas["logistic_regression"]["step2_vs_baseline"]["point_delta"] >= 0.01,
    final_deltas["logistic_regression"]["step3_vs_step2"]["significant"]
    and final_deltas["logistic_regression"]["step3_vs_step2"]["point_delta"] >= 0.01,
)
xgb_ranking = (
    final_deltas["xgboost"]["step2_vs_baseline"]["significant"]
    and final_deltas["xgboost"]["step2_vs_baseline"]["point_delta"] >= 0.01,
    final_deltas["xgboost"]["step3_vs_step2"]["significant"]
    and final_deltas["xgboost"]["step3_vs_step2"]["point_delta"] >= 0.01,
)
robustness_holds_on_test = logreg_ranking == xgb_ranking
print(f"\nRobustness (same ranking on the held-out test set): {robustness_holds_on_test}")



=== logistic_regression ===
Test ROC-AUC by step: baseline=0.8640, step2_rate_level=0.8648, step3_rate_change=0.8614
Test PR-AUC by step:  baseline=0.5094, step2_rate_level=0.5154, step3_rate_change=0.4797  (test-window baseline prevalence: 0.0853, i.e. the PR-AUC floor for this window)
  Step 2 vs baseline: delta +0.0008, 95% CI [-0.0027, +0.0039] -> NO SIGNIFICANT MARGINAL CONTRIBUTION (95% CI includes zero)
  Step 3 vs step 2  : delta -0.0034, 95% CI [-0.0222, +0.0186] -> NO SIGNIFICANT MARGINAL CONTRIBUTION (95% CI includes zero)

=== xgboost ===
Test ROC-AUC by step: baseline=0.8822, step2_rate_level=0.8731, step3_rate_change=0.8747
Test PR-AUC by step:  baseline=0.4615, step2_rate_level=0.4242, step3_rate_change=0.4466  (test-window baseline prevalence: 0.0853, i.e. the PR-AUC floor for this window)
  Step 2 vs baseline: delta -0.0092, 95% CI [-0.0191, -0.0003] -> significant, but below the +0.01 threshold
  Step 3 vs step 2  : delta +0.0016, 95% CI [-0.0077, +0.0102] -> NO SIGN

## 9. Order-dependency statement

Per `docs/phase0_framing.md`: the marginal-lift numbers above depend on the
order features were added in (momentum first). They represent what
mortgage-rate movement contributes *given that a county's own price
momentum is already known*, not a claim about rate's overall importance in
isolation. A different ordering (rate first, momentum added on top) would
be expected to attribute a different, larger share of predictive power to
rate, since momentum would no longer be "already known" when rate's
contribution is measured -- that alternative ordering was not run, and this
project's result should not be read as if it had been.


## 10. Persist the final evaluation

In [9]:
final_results = {
    "test_window": {"start": str(TEST_START.date()), "end": str(TEST_END.date())},
    "test_data_quality_gate": {
        "observed_rows": int(observed_test_rows),
        "observed_positive_rate_pct": float(observed_test_rate),
        "expected_rows_per_phase0_framing": expected_test_rows,
        "expected_positive_rate_pct_per_phase0_framing": expected_test_rate,
    },
    "final_scores_full_feature_set": final_scores,
    "test_step_scores": test_step_scores,
    "test_deltas": final_deltas,
    "robustness_holds_on_test": bool(robustness_holds_on_test),
    "n_bootstrap": N_BOOTSTRAP,
    "random_state": RANDOM_STATE,
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/final_evaluation.json", "w") as fh:
    json.dump(final_results, fh, indent=2)
print("Saved outputs/final_evaluation.json")


Saved outputs/final_evaluation.json


## 11. Findings


---

## Final write-up

**The headline result, from the run recorded below.** A momentum-only baseline
predicts next-quarter decline risk well on its own: test ROC-AUC of 0.8640
(logistic regression) and 0.8822 (XGBoost), well above chance, and PR-AUC of
0.51 and 0.46 respectively against this test window's own 8.53% positive-rate
floor. Adding the mortgage rate's level on top of momentum (step 2) does not
clear this project's ROC-AUC >= baseline + 0.01 threshold: logistic
regression's delta is +0.0008 with a 95% CI of [-0.0027, +0.0039] (no
significant marginal contribution), and XGBoost's delta is -0.0092 with a 95%
CI of [-0.0191, -0.0003] (a small but statistically significant *decrease*).
Adding rate-of-change and the interaction term on top of that (step 3) also
shows no significant marginal contribution for either algorithm (logistic
regression: -0.0034, CI [-0.0222, +0.0186]; XGBoost: +0.0016, CI [-0.0077,
+0.0102]). The robustness check passed: the same ranking of contributions
(momentum carries the real signal; neither rate feature adds anything
significant) held across both algorithm families.

**This is a stated null result on rate's marginal contribution, and
`docs/phase0_framing.md`'s own Success Criteria table explicitly allows for
that as a valid, reportable outcome.** A county's own price momentum does the
real predictive work. Once that's already known, neither the mortgage rate's
level nor its rate of change adds measurable predictive power on top of it --
and for XGBoost, adding the rate level measurably *hurt* performance by a
small amount. That's a specific, defensible answer to this project's business
question, not an absence of one.

**A note on the rate-of-change sign mismatch, and what notebook 11 found.**
Notebook `06_eda.ipynb`'s lag sweep found that the mortgage rate's
rate-of-change is most correlated with next-quarter decline risk at a 12-month
lag, but with a negative sign (r = -0.0521, 95% CI [-0.0568, -0.0474]), the
opposite of the positive sign the stated hypothesis expected (rising rates ->
higher borrowing cost -> more decline risk). This was flagged at the time as a
face-validity surprise rather than smoothed over. That interval treats about
170,000 county-quarter rows as independent, but the rate is one national
number per quarter; tested at quarter level (83 quarters, notebook 06 Section
9b), the correlation is r = -0.165 with a 95% CI of [-0.368, +0.052], so the
sign itself is not reliable. One possible explanation for a negative sign is a
feedback loop: the Federal Reserve does not set rates in a vacuum, and a
cooling housing market may be followed by rate cuts.
`notebooks/11_reverse_causation_check.ipynb` tested this by correlating
national home-price momentum against *future* mortgage rate-of-change across
18 forward lags. Using only rates through the end of Train, the best lag is
4 months (r = 0.124, n = 254), and randomly shifted rate series produce a best
correlation at least that large 39% of the time (best-of-18 p = 0.39), so the
training-period data show no relationship beyond chance. The stronger
14-month result in the first version of notebook 11 (r = 0.2637) depends on
rates from after the end of Train, from a single episode (the 2020-21 price
boom followed by the 2022 rate hikes), which cannot separate a housing
feedback loop from the Fed reacting to general inflation. So the feedback-loop
explanation is not supported, and remains a possibility rather than a finding:
there is no reliable evidence that home prices lead mortgage rates. See
`outputs/reverse_causation_check.json` for both variants. This does not change
this project's headline result (rate-of-change still showed no significant
marginal contribution once momentum is already known, so the sign question
does not affect the modeling conclusion either way).

**Why these numbers, and not something close to a perfect ROC-AUC.** An
earlier version of this pipeline had a label/feature alignment bug: the
momentum features and the decline label were both computed as the price
change ending at the *same* quarter-end, so `momentum_3m` and `decline_label`
were, for practical purposes, measuring the same underlying quantity. That
produced a baseline ROC-AUC of 0.99998 -- not a good result, a near-certain
symptom of the model being handed a continuous version of the answer rather
than forecasting anything. The fix, in `src/housing_risk/features.py`'s
`build_feature_matrix`, shifts the label onto the *prior* quarter's features,
so a row's features now reflect only what was knowable as of that quarter,
and its label describes the outcome one quarter later -- matching this
project's business question, which explicitly asks about predicting one
quarter ahead. Every number in this notebook, and in notebooks 07 through 09,
reflects that fix.

**Why the test-window row count and positive rate differ slightly from
`docs/phase0_framing.md`'s originally documented 52,087 rows / 8.15%.** That
document's figures were computed before the alignment fix above, using a
same-quarter join that had no trailing-edge exclusion. After the fix, the
single most recent quarter-end in any loaded window has no "one quarter
later" price data available to compute its label from (that data would fall
outside the window on purpose), so that quarter's rows are correctly dropped
project-wide, in notebook 07's training matrix and in this notebook's test
matrix alike. Section 3 and Section 4 above name this mechanism and print the
exact row count it drops, rather than leaving the shifted figures as an
unexplained discrepancy for someone to puzzle over later.



- **This notebook is the first and only place the held-out test set
  (2022-01-01 to 2026-03-31) is scored** in this project -- every prior
  modeling notebook worked within the training period only.
- **The per-window data-quality gate compared the test window's observed
  positive rate against its own documented 8.15% figure** (Section 4), not
  a blanket tolerance shared with the training window's different (4.96%)
  rate.
- **PR-AUC is read against the test window's own baseline prevalence**
  (Section 8), not the full-history 5.59% rate, since the test window's
  positive rate is meaningfully higher.
- **Every marginal AUC delta on the test set carries a quarter-clustered
  bootstrap 95% CI** (Section 7), the same method used throughout notebooks
  08 and 09, so training-period CV results and this final test-set result
  are comparable in method even though they are never compared to each
  other as if they were the same number (per `docs/phase0_framing.md`: CV
  deltas while developing, test-set deltas with bootstrap CIs at final
  evaluation -- not one substituted for the other).
- **The robustness finding is re-checked on held-out data** (Section 8),
  not assumed to carry over unchanged from notebooks 08/09's training-period
  CV result.
- **The order-dependency of every marginal lift is stated explicitly**
  (Section 9): these numbers answer what rate adds *given* momentum is
  already known, not rate's importance in isolation.
